# S-DoT 유동인구 시기별 분석

**목적**: 기존 전체기간(632일) 일평균 분석을 **년도별/계절별/월별**로 세분화

**데이터**: S-DoT 유동인구 + 외국인 생활인구 (2024.01 ~ 2025.09)

**각 기간별 5가지 분석**:
1. 자치구별 유동인구 TOP 10
2. 지역유형별 유동인구
3. 시간대별 유동인구 (피크타임)
4. 핵심 관광지 동별 유동인구
5. 복합점수 (외국인 + 유동인구 → Hub/Spoke)

**집계 공식**:
- S-DoT 일평균 방문자 = 해당 기간 방문자수 합계 / 해당 기간 일수
- 외국인 일평균 = Σ(10~22시 외국인) / 13시간 / 해당 기간 일수 (방법 B)
- 복합점수 = MinMax(외국인_일평균) + MinMax(S-DoT_일평균) (범위 0~2)
- Hub/Spoke 기준 = 복합점수 상위 30%

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings('ignore')

# 경로 설정
BASE = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '06_analysis', '01_foreigner'))
RAW_SDOT = os.path.join(BASE, '01_raw_data', 'S-DoT_WALK')
RAW_FOREIGNER = os.path.join(BASE, '01_raw_data')
PROCESSED = os.path.join(BASE, '02_processed_data')
OUTPUT_DIR = os.path.join(PROCESSED, 'temporal')

DATE_START = '2024-01-01'
DATE_END = '2025-09-30'
DAISO_HOURS = list(range(10, 23))
NUM_HOURS = len(DAISO_HOURS)

# 매핑 테이블
SDOT_GU_MAP = {
    'Jongno-gu': '종로구', 'Jung-gu': '중구', 'Yongsan-gu': '용산구',
    'Seongdong-gu': '성동구', 'Gwangjin-gu': '광진구', 'Dongdaemun-gu': '동대문구',
    'Jungnang-gu': '중랑구', 'Seongbuk-gu': '성북구', 'Gangbuk-gu': '강북구',
    'Dobong-gu': '도봉구', 'Nowon-gu': '노원구', 'Eunpyeong-gu': '은평구',
    'Seodaemun-gu': '서대문구', 'Mapo-gu': '마포구', 'Yangcheon-gu': '양천구',
    'Gangseo-gu': '강서구', 'Guro-gu': '구로구', 'Geumcheon-gu': '금천구',
    'Yeongdeungpo-gu': '영등포구', 'Dongjak-gu': '동작구', 'Gwanak-gu': '관악구',
    'Seocho-gu': '서초구', 'Gangnam-gu': '강남구', 'Songpa-gu': '송파구',
    'Gangdong-gu': '강동구'
}
GU_CODE_MAP = {
    '11110': '종로구', '11140': '중구', '11170': '용산구', '11200': '성동구',
    '11215': '광진구', '11230': '동대문구', '11260': '중랑구', '11290': '성북구',
    '11305': '강북구', '11320': '도봉구', '11350': '노원구', '11380': '은평구',
    '11410': '서대문구', '11440': '마포구', '11470': '양천구', '11500': '강서구',
    '11530': '구로구', '11545': '금천구', '11560': '영등포구', '11590': '동작구',
    '11620': '관악구', '11650': '서초구', '11680': '강남구', '11710': '송파구',
    '11740': '강동구'
}
TYPE_MAP = {
    'main_street': '주요 거리', 'traditional_markets': '전통시장',
    'parks': '공원', 'commercial_area': '상업지역',
    'residential_area': '주거지역', 'public_facilities': '공공시설'
}
TOURIST_DONGS = {
    'Myeong-dong': '명동', 'Gwanghui-dong': '광희동(DDP)',
    'Hoehyeon-dong': '회현동(남대문)', 'Sinsa-dong': '신사동',
    'Gahoe-dong': '가회동(북촌)', 'Apgujeong-dong': '압구정동',
    'Samcheong-dong': '삼청동', 'Itaewon2-dong': '이태원2동'
}
SEASON_MAP = {
    1: '겨울', 2: '겨울', 3: '봄', 4: '봄', 5: '봄',
    6: '여름', 7: '여름', 8: '여름', 9: '가을', 10: '가을',
    11: '가을', 12: '겨울'
}

print('설정 완료')

## 1. 데이터 로드 및 전처리

In [ ]:
# S-DoT 유동인구 로드
sdot_files = sorted(glob.glob(os.path.join(RAW_SDOT, 'S-DoT_WALK_*.csv')))
print(f'[S-DoT] 파일 수: {len(sdot_files)}')

dfs = []
for f in sdot_files:
    for enc in ['utf-8', 'cp949', 'euc-kr']:
        try:
            df = pd.read_csv(f, encoding=enc)
            dfs.append(df)
            break
        except:
            continue

sdot_raw = pd.concat(dfs, ignore_index=True)
print(f'[S-DoT] 총 레코드: {len(sdot_raw):,}')
sdot_raw.head(3)

In [ ]:
# 외국인 생활인구 로드 (2024.01~2025.09)
target_months = []
for y in [2024, 2025]:
    end_m = 12 if y == 2024 else 9
    for m in range(1, end_m + 1):
        target_months.append(f'TEMP_FOREIGNER_{y}{m:02d}')

fg_files = []
for folder in target_months:
    folder_path = os.path.join(RAW_FOREIGNER, folder)
    if os.path.exists(folder_path):
        fg_files.extend(glob.glob(os.path.join(folder_path, 'TEMP_FOREIGNER_*.csv')))

print(f'[외국인] 파일 수: {len(fg_files)}')

fg_dfs = []
for f in sorted(fg_files):
    try:
        df = pd.read_csv(f, encoding='cp949')
        df.columns = [col.replace('\ufeff', '').replace('?', '').strip('"') for col in df.columns]
        fg_dfs.append(df)
    except:
        pass

fg_raw = pd.concat(fg_dfs, ignore_index=True)
print(f'[외국인] 총 레코드: {len(fg_raw):,}')

In [ ]:
# S-DoT 전처리
sdot = sdot_raw.copy()
sdot['자치구_한글'] = sdot['자치구'].map(SDOT_GU_MAP)
sdot['방문자수'] = pd.to_numeric(sdot['방문자수'], errors='coerce').fillna(0)
sdot['측정일'] = sdot['측정시간'].str[:10]
sdot['시간'] = sdot['측정시간'].str[11:13].astype(int)

sdot = sdot[sdot['시간'].between(10, 22)].copy()
sdot['측정일_dt'] = pd.to_datetime(sdot['측정일'], errors='coerce')
sdot = sdot[(sdot['측정일_dt'] >= DATE_START) & (sdot['측정일_dt'] <= DATE_END)].copy()
sdot = sdot[sdot['자치구_한글'].notna()].copy()

sdot['년도'] = sdot['측정일_dt'].dt.year
sdot['월'] = sdot['측정일_dt'].dt.month
sdot['년월'] = sdot['측정일_dt'].dt.strftime('%Y-%m')
sdot['계절'] = sdot['월'].map(SEASON_MAP)

print(f'[S-DoT] 최종: {len(sdot):,}건, {sdot["측정일"].nunique()}일')

In [ ]:
# 외국인 전처리
fg = fg_raw.copy()

col_renames = {}
for col in fg.columns:
    if '기준일' in col: col_renames[col] = '기준일'
    elif '시간대' in col: col_renames[col] = '시간대'
    elif '행정동코드' in col: col_renames[col] = '행정동코드'
    elif '총생활인구수' in col: col_renames[col] = '총생활인구수'
    elif '중국인체류인구수' in col: col_renames[col] = '중국인체류인구수'
    elif '중국외외국인체류인구수' in col: col_renames[col] = '중국외외국인체류인구수'
fg.rename(columns=col_renames, inplace=True)

fg['시간대'] = pd.to_numeric(fg['시간대'], errors='coerce')
fg = fg[fg['시간대'].isin(DAISO_HOURS)].copy()

for col in ['중국인체류인구수', '중국외외국인체류인구수']:
    fg[col] = pd.to_numeric(fg[col].replace('*', np.nan), errors='coerce').fillna(0)
fg['총생활인구수'] = pd.to_numeric(fg['총생활인구수'], errors='coerce').fillna(0)
fg['외국인체류인구수'] = fg['중국인체류인구수'] + fg['중국외외국인체류인구수']

fg['행정동코드'] = fg['행정동코드'].astype(str)
fg['구코드'] = fg['행정동코드'].str[:5]
fg['자치구'] = fg['구코드'].map(GU_CODE_MAP)
fg['기준일'] = fg['기준일'].astype(str)
fg['날짜'] = pd.to_datetime(fg['기준일'], format='%Y%m%d', errors='coerce')
fg = fg[(fg['날짜'] >= DATE_START) & (fg['날짜'] <= DATE_END)].copy()
fg = fg[fg['자치구'].notna()].copy()

fg['년도'] = fg['날짜'].dt.year
fg['월'] = fg['날짜'].dt.month
fg['년월'] = fg['날짜'].dt.strftime('%Y-%m')
fg['계절'] = fg['월'].map(SEASON_MAP)

print(f'[외국인] 최종: {len(fg):,}건, {fg["기준일"].nunique()}일')

## 2. 분석 함수 정의

In [ ]:
def analyze_gu(sdot_df, days):
    """자치구별 유동인구"""
    agg = sdot_df.groupby('자치구_한글').agg(
        방문자수_합=('방문자수', 'sum'), 센서수=('시리얼', 'nunique')
    ).reset_index()
    agg.rename(columns={'자치구_한글': '자치구'}, inplace=True)
    agg['일평균_방문자'] = (agg['방문자수_합'] / days).round(0)
    agg['센서당_일평균'] = (agg['일평균_방문자'] / agg['센서수']).round(0)
    return agg.sort_values('일평균_방문자', ascending=False)


def analyze_type(sdot_df, days):
    """지역유형별 유동인구"""
    agg = sdot_df.groupby('지역').agg(
        방문자수_합=('방문자수', 'sum'), 센서수=('시리얼', 'nunique')
    ).reset_index()
    agg['유형'] = agg['지역'].map(TYPE_MAP)
    agg['일평균_방문자'] = (agg['방문자수_합'] / days).round(0)
    agg['센서당_일평균'] = (agg['일평균_방문자'] / agg['센서수']).round(0)
    return agg.sort_values('일평균_방문자', ascending=False)


def analyze_hourly(sdot_df, days):
    """시간대별 유동인구"""
    agg = sdot_df.groupby('시간').agg(방문자수_합=('방문자수', 'sum')).reset_index()
    agg['일평균_방문자'] = (agg['방문자수_합'] / days).round(0)
    peak = agg.loc[agg['일평균_방문자'].idxmax()]
    return agg, int(peak['시간']), peak['일평균_방문자']


def analyze_tourist_dong(sdot_df, days):
    """핵심 관광지 동별 유동인구"""
    df_tour = sdot_df[sdot_df['행정동'].isin(TOURIST_DONGS.keys())].copy()
    if len(df_tour) == 0:
        return pd.DataFrame()
    agg = df_tour.groupby(['자치구_한글', '행정동']).agg(
        방문자수_합=('방문자수', 'sum'), 센서수=('시리얼', 'nunique')
    ).reset_index()
    agg.rename(columns={'자치구_한글': '자치구'}, inplace=True)
    agg['동_한글'] = agg['행정동'].map(TOURIST_DONGS)
    agg['일평균_방문자'] = (agg['방문자수_합'] / days).round(0)
    hourly = df_tour.groupby(['행정동', '시간'])['방문자수'].sum().reset_index()
    hourly['일평균'] = (hourly['방문자수'] / days).round(0)
    peak_hours = hourly.loc[hourly.groupby('행정동')['일평균'].idxmax()][['행정동', '시간']].rename(columns={'시간': '피크시간'})
    agg = agg.merge(peak_hours, on='행정동', how='left')
    return agg.sort_values('일평균_방문자', ascending=False)


def analyze_composite(sdot_df, foreigner_df, sdot_days, fg_days):
    """복합점수 (외국인 + 유동인구)"""
    sdot_gu = sdot_df.groupby('자치구_한글')['방문자수'].sum().reset_index()
    sdot_gu.rename(columns={'자치구_한글': '자치구', '방문자수': 'sdot_합'}, inplace=True)
    sdot_gu['S-DoT_일평균'] = (sdot_gu['sdot_합'] / sdot_days).round(0)

    fg_gu = foreigner_df.groupby('자치구')['외국인체류인구수'].sum().reset_index()
    fg_gu['외국인_일평균'] = (fg_gu['외국인체류인구수'] / NUM_HOURS / fg_days).round(0)

    merged = pd.merge(fg_gu[['자치구', '외국인_일평균']], sdot_gu[['자치구', 'S-DoT_일평균']], on='자치구', how='inner')
    if len(merged) < 2:
        return merged

    for col, new_col in [('외국인_일평균', '외국인_정규화'), ('S-DoT_일평균', '유동량_정규화')]:
        min_v, max_v = merged[col].min(), merged[col].max()
        merged[new_col] = ((merged[col] - min_v) / (max_v - min_v + 1e-10)).round(4)

    merged['복합점수'] = (merged['외국인_정규화'] + merged['유동량_정규화']).round(4)
    threshold = merged['복합점수'].quantile(0.70)
    merged['분류'] = merged['복합점수'].apply(lambda x: 'Hub' if x >= threshold else 'Spoke')
    return merged.sort_values('복합점수', ascending=False)

print('분석 함수 정의 완료')

## 3. 년도별 분석 (2024 vs 2025)

> **⚠️ 주의**: 2024년은 12개월(359일), 2025년은 1~9월(273일)로 기간이 다릅니다.
> 2024년 10월이 전 기간 최고 피크(132,797명/일)여서 단순 비교 시 왜곡이 발생합니다.
> **공정한 비교는 아래 "3-1. 동일 기간(1~9월) 비교"를 참고하세요.**

In [ ]:
# 년도별 자치구 유동인구
for year in [2024, 2025]:
    s = sdot[sdot['년도'] == year]
    days = s['측정일'].nunique()
    r = analyze_gu(s, days)
    print(f'\n{"=" * 65}')
    print(f'{year}년 자치구별 유동인구 TOP 10 ({days}일)')
    print(f'집계: 방문자수 합계 / {days}일')
    print(f'{"=" * 65}')
    print(r[['자치구', '일평균_방문자', '센서수', '센서당_일평균']].head(10).to_string(index=False))

In [ ]:
# 년도별 지역유형별
for year in [2024, 2025]:
    s = sdot[sdot['년도'] == year]
    days = s['측정일'].nunique()
    r = analyze_type(s, days)
    print(f'\n{"=" * 55}')
    print(f'{year}년 지역유형별 유동인구 ({days}일)')
    print(f'{"=" * 55}')
    print(r[['유형', '일평균_방문자', '센서수', '센서당_일평균']].to_string(index=False))

In [ ]:
# 년도별 시간대별
for year in [2024, 2025]:
    s = sdot[sdot['년도'] == year]
    days = s['측정일'].nunique()
    r, peak_h, peak_v = analyze_hourly(s, days)
    print(f'\n{"=" * 50}')
    print(f'{year}년 시간대별 유동인구 ({days}일) — 피크: {peak_h}시 ({peak_v:,.0f}명)')
    print(f'{"=" * 50}')
    print(r[['시간', '일평균_방문자']].to_string(index=False))

In [ ]:
# 년도별 관광지 동별
for year in [2024, 2025]:
    s = sdot[sdot['년도'] == year]
    days = s['측정일'].nunique()
    r = analyze_tourist_dong(s, days)
    print(f'\n{"=" * 75}')
    print(f'{year}년 핵심 관광지 동별 유동인구 ({days}일)')
    print(f'{"=" * 75}')
    if len(r) > 0:
        print(r[['자치구', '동_한글', '일평균_방문자', '피크시간']].to_string(index=False))

In [ ]:
# 년도별 복합점수
for year in [2024, 2025]:
    s = sdot[sdot['년도'] == year]
    f = fg[fg['년도'] == year]
    s_days = s['측정일'].nunique()
    f_days = f['기준일'].nunique()
    r = analyze_composite(s, f, s_days, f_days)
    print(f'\n{"=" * 85}')
    print(f'{year}년 복합점수 (S-DoT {s_days}일, 외국인 {f_days}일)')
    print(f'집계: MinMax(외국인 Σ/13h/{f_days}일) + MinMax(S-DoT Σ/{s_days}일)')
    print(f'{"=" * 85}')
    print(r[['자치구', '외국인_일평균', 'S-DoT_일평균', '복합점수', '분류']].to_string(index=False))
    hub = r[r['분류'] == 'Hub']['자치구'].tolist()
    print(f'\nHub: {hub}')

## 3-1. 동일 기간(1~9월) 비교 분석

> **⚠️ 2024년은 12개월(359일), 2025년은 9개월(273일)**로 포함 기간이 다릅니다.
> 2024년 10월이 전 기간 최고 피크(132,797명/일)여서, 단순 년도 비교 시 계절 구성 차이로 왜곡이 발생합니다.
> 아래는 **동일 기간(1~9월)**만 추출하여 공정하게 비교한 결과입니다.

In [ ]:
# 동일 기간(1~9월) 자치구별 비교
print('=' * 80)
print('[동일 기간 비교] 자치구별 유동인구 (1~9월)')
print('=' * 80)

for year in [2024, 2025]:
    s = sdot[(sdot['년도'] == year) & (sdot['월'] <= 9)]
    days = s['측정일'].nunique()
    r = analyze_gu(s, days)
    print(f'\n{year}년 1~9월 ({days}일) TOP 10:')
    print(r[['자치구', '일평균_방문자', '센서수', '센서당_일평균']].head(10).to_string(index=False))

# 증감 비교
s24 = sdot[(sdot['년도'] == 2024) & (sdot['월'] <= 9)]
s25 = sdot[(sdot['년도'] == 2025) & (sdot['월'] <= 9)]
d24, d25 = s24['측정일'].nunique(), s25['측정일'].nunique()
r24 = analyze_gu(s24, d24).set_index('자치구')
r25 = analyze_gu(s25, d25).set_index('자치구')
compare = pd.DataFrame({
    '2024_일평균': r24['일평균_방문자'],
    '2025_일평균': r25['일평균_방문자']
}).dropna()
compare['증감(%)'] = ((compare['2025_일평균'] / compare['2024_일평균'] - 1) * 100).round(1)
print(f'\n{"=" * 50}')
print('[동일 기간] 자치구별 증감 TOP 10:')
print(compare.sort_values('2024_일평균', ascending=False).head(10).to_string())

In [ ]:
# 동일 기간(1~9월) 관광지 동별 비교 — 핵심!
print('=' * 80)
print('[동일 기간 비교] 핵심 관광지 동별 유동인구 (1~9월)')
print('=' * 80)

for year in [2024, 2025]:
    s = sdot[(sdot['년도'] == year) & (sdot['월'] <= 9)]
    days = s['측정일'].nunique()
    r = analyze_tourist_dong(s, days)
    print(f'\n{year}년 1~9월 ({days}일):')
    if len(r) > 0:
        print(r[['자치구', '동_한글', '일평균_방문자', '피크시간']].to_string(index=False))

# 관광지 증감
s24 = sdot[(sdot['년도'] == 2024) & (sdot['월'] <= 9)]
s25 = sdot[(sdot['년도'] == 2025) & (sdot['월'] <= 9)]
d24, d25 = s24['측정일'].nunique(), s25['측정일'].nunique()
t24 = analyze_tourist_dong(s24, d24).set_index('동_한글')
t25 = analyze_tourist_dong(s25, d25).set_index('동_한글')
compare_t = pd.DataFrame({
    '2024_일평균': t24['일평균_방문자'],
    '2025_일평균': t25['일평균_방문자']
}).dropna()
compare_t['증감(%)'] = ((compare_t['2025_일평균'] / compare_t['2024_일평균'] - 1) * 100).round(1)
print(f'\n{"=" * 50}')
print('[핵심 인사이트] 동일기간 관광지 증감:')
print(compare_t.sort_values('2024_일평균', ascending=False).to_string())
print('\n→ 명동, DDP, 북촌, 삼청동은 동일기간 비교 시 오히려 증가!')

In [ ]:
# 동일 기간(1~9월) 복합점수 비교
print('=' * 80)
print('[동일 기간 비교] 복합점수 (1~9월)')
print('=' * 80)

for year in [2024, 2025]:
    s = sdot[(sdot['년도'] == year) & (sdot['월'] <= 9)]
    f = fg[(fg['년도'] == year) & (fg['월'] <= 9)]
    s_days = s['측정일'].nunique()
    f_days = f['기준일'].nunique()
    r = analyze_composite(s, f, s_days, f_days)
    print(f'\n{year}년 1~9월 (S-DoT {s_days}일, 외국인 {f_days}일)')
    print(r[['자치구', '외국인_일평균', 'S-DoT_일평균', '복합점수', '분류']].to_string(index=False))
    hub = r[r['분류'] == 'Hub']['자치구'].tolist()
    print(f'Hub: {hub}')

print('\n' + '=' * 50)
print('[결론] 동일기간에서도 Hub TOP 3 (중구·강남구·동작구)는 동일')
print('→ 구조적으로 안정적인 Hub임을 확인')

## 4. 계절별 분석 (봄/여름/가을/겨울)

In [ ]:
# 계절별 자치구 TOP 5
for season in ['봄', '여름', '가을', '겨울']:
    s = sdot[sdot['계절'] == season]
    days = s['측정일'].nunique()
    r = analyze_gu(s, days)
    print(f'\n[{season}] TOP 5 ({days}일)')
    print(r[['자치구', '일평균_방문자', '센서당_일평균']].head(5).to_string(index=False))

In [ ]:
# 계절별 피크타임
print(f'{"계절":<6} {"피크시간":<8} {"일평균_방문자":>12}')
print('-' * 30)
for season in ['봄', '여름', '가을', '겨울']:
    s = sdot[sdot['계절'] == season]
    days = s['측정일'].nunique()
    _, peak_h, peak_v = analyze_hourly(s, days)
    print(f'{season:<6} {peak_h}시{"":<4} {peak_v:>12,.0f}')

In [ ]:
# 계절별 명동·DDP 비교
print(f'{"계절":<6} {"명동":>10} {"DDP":>10} {"북촌":>10} {"신사":>10}')
print('-' * 50)
for season in ['봄', '여름', '가을', '겨울']:
    s = sdot[sdot['계절'] == season]
    days = s['측정일'].nunique()
    r = analyze_tourist_dong(s, days)
    if len(r) == 0:
        continue
    vals = {}
    for _, row in r.iterrows():
        vals[row['동_한글']] = row['일평균_방문자']
    print(f'{season:<6} {vals.get("명동", 0):>10,.0f} {vals.get("광희동(DDP)", 0):>10,.0f} {vals.get("가회동(북촌)", 0):>10,.0f} {vals.get("신사동", 0):>10,.0f}')

In [ ]:
# 계절별 복합점수 Hub
for season in ['봄', '여름', '가을', '겨울']:
    s = sdot[sdot['계절'] == season]
    f = fg[fg['계절'] == season]
    s_days = s['측정일'].nunique()
    f_days = f['기준일'].nunique()
    r = analyze_composite(s, f, s_days, f_days)
    hub = r[r['분류'] == 'Hub'][['자치구', '복합점수']].head(3)
    hub_str = ', '.join([f"{row['자치구']}({row['복합점수']:.2f})" for _, row in hub.iterrows()])
    print(f'[{season}] Hub TOP 3: {hub_str}')

## 5. 월별 분석 (2024.01 ~ 2025.09)

In [ ]:
# 월별 요약
months = sorted(sdot['년월'].unique())
print(f'{"월":<8} {"일수":>4} {"TOP1":<6} {"TOP2":<6} {"TOP3":<6} {"피크":>4} {"Hub TOP 3"}')
print('-' * 65)

for ym in months:
    s = sdot[sdot['년월'] == ym]
    f = fg[fg['년월'] == ym]
    s_days = s['측정일'].nunique()
    f_days = f['기준일'].nunique()
    
    gu = analyze_gu(s, s_days)
    top3 = gu.head(3)['자치구'].tolist()
    _, peak_h, _ = analyze_hourly(s, s_days)
    
    if f_days > 0:
        comp = analyze_composite(s, f, s_days, f_days)
        hub = comp[comp['분류'] == 'Hub']['자치구'].head(3).tolist()
    else:
        hub = []
    
    print(f'{ym:<8} {s_days:>4} {top3[0] if len(top3)>0 else "":<6} {top3[1] if len(top3)>1 else "":<6} {top3[2] if len(top3)>2 else "":<6} {peak_h:>2}시 {", ".join(hub)}')

## 6. 교차검증

In [ ]:
# 년도별 합산 검증
total_sdot_days = sdot['측정일'].nunique()
y2024_days = sdot[sdot['년도']==2024]['측정일'].nunique()
y2025_days = sdot[sdot['년도']==2025]['측정일'].nunique()
print(f'S-DoT 일수: {y2024_days} + {y2025_days} = {y2024_days+y2025_days} (전체 {total_sdot_days})')

total_fg_days = fg['기준일'].nunique()
y2024_fg = fg[fg['년도']==2024]['기준일'].nunique()
y2025_fg = fg[fg['년도']==2025]['기준일'].nunique()
print(f'외국인 일수: {y2024_fg} + {y2025_fg} = {y2024_fg+y2025_fg} (전체 {total_fg_days})')

total_v = sdot['방문자수'].sum()
y2024_v = sdot[sdot['년도']==2024]['방문자수'].sum()
y2025_v = sdot[sdot['년도']==2025]['방문자수'].sum()
print(f'\n방문자 합산: {y2024_v:,.0f} + {y2025_v:,.0f} = {y2024_v+y2025_v:,.0f}')
print(f'전체 합산: {total_v:,.0f}')
print(f'일치 여부: {abs(total_v - (y2024_v+y2025_v)) < 1}')

In [ ]:
# 저장된 CSV 확인
csv_files = sorted(os.listdir(OUTPUT_DIR))
print(f'저장된 CSV 파일 수: {len(csv_files)}개')
print(f'경로: {OUTPUT_DIR}')
for f in csv_files[:10]:
    print(f'  {f}')
if len(csv_files) > 10:
    print(f'  ... 외 {len(csv_files)-10}개')